In [1]:
import pandas as pd

from data_preprocessingagent import DataPreprocessingAgent
from health_monitoring_agent import HealthMonitoringAgent
from fault_detection_agent import FaultDetectionAgent

In [3]:
pre_agent=DataPreprocessingAgent()
train,test,rul=pre_agent.preprocess(
    "train_FD003.txt",
    "test_FD003.txt",
    "RUL_FD003.txt"
)
health_agent=HealthMonitoringAgent()
health_df=health_agent.monitor(train)
fault_agent=FaultDetectionAgent()
fault_df=fault_agent.detect(health_df)

Shape: (24720, 26)
Missing: 0
Duplicates: 0


In [4]:
risk_df=fault_df.copy()

In [6]:
### Calculate risk score

In [5]:
risk_df["Risk_Score"]=(
    (100-risk_df["Health_Score"])*0.5
    +
    risk_df["Anomaly_Count"]*5
    +
    risk_df["Degradation_Score"]*20
)

### Normalize risk score

In [7]:
max_risk=risk_df["Risk_Score"].max()
risk_df["Risk_Score"]=(risk_df["Risk_Score"]/max_risk)*100

In [8]:
def get_risk_level(score):
    if score>=70:
        return "High"
    elif score>=40:
        return "Medium"
    else:
        return "Low"

In [9]:
risk_df["Risk_Level"]=(risk_df["Risk_Score"].apply(get_risk_level))

### upgrade faults to high risk

In [10]:
risk_df.loc[
    risk_df["Fault_Status"] == "Fault Detected",
    "Risk_Level"
]="High"

In [11]:
risk_df[
    [
        "engine_id",
        "Health_Score",
        "Fault_Status",
        "Risk_Score",
        "Risk_Level"
    ]
].head(20)

,engine_id,Health_Score,Fault_Status,Risk_Score,Risk_Level
0,1,100.000000,Normal,0.000000,Low
1,1,97.655816,Normal,1.669182,Low
2,1,97.013170,Normal,2.126779,Low
3,1,96.452752,Normal,2.525827,Low
4,1,96.035272,Normal,2.823095,Low
5,1,96.508289,Normal,2.486281,Low
6,1,96.836692,Normal,2.252441,Low
7,1,97.507475,Normal,1.774809,Low
8,1,96.096874,Normal,2.779230,Low
9,1,96.960142,Normal,2.164538,Low


In [12]:
risk_df["Risk_Level"].value_counts()

Risk_Level
Low       23801
High        837
Medium       82
Name: count, dtype: int64

In [13]:
risk_df.to_csv(
    "risk_analysis_output.csv",
    index=False
)
print("Risk Analysis Output Saved")

Risk Analysis Output Saved
